# CALE Psychometric Audit Benchmark

This notebook extends `strong_evaluator_results.ipynb` into a psychometric audit pipeline.

It is designed to answer four questions:

1. **Internal structure**: Do CALE dimensions form a coherent but non-redundant measurement structure?
2. **Evaluator psychometric quality**: Which evaluator backend better conforms to a theoretically motivated measurement model?
3. **Convergent / discriminant evidence**: Do evaluator scores align with construct-relevant signals and remain separable from nuisance signals?
4. **Target-model ability profiles**: Can CALE-derived latent factor scores produce interpretable, evaluator-mediated ability profiles for the evaluated models?

## Claim boundary

This notebook provides **internal psychometric audit evidence**, not final proof of construct validity. CFA fit, factor scores, and proxy-based convergent/discriminant diagnostics should be described as measurement-behavior evidence unless external human validation labels are added.


## 0. File and format assumptions

This notebook expects the same project layout and CSV formats as the previous notebook `strong_evaluator_results.ipynb`.

Expected input files:

| run_id | expected file |
|---|---|
| `heuristic_full` | `outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_full_eval_behavior_matrix.csv` |
| `qwen25_7b_limit1000` | `outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_strong_qwen25_7b_limit1000_eval_behavior_matrix.csv` |
| `qwen25_7b_llama_limit1000` | `outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_strong_qwen25_7b_llama_limit1000_eval_behavior_matrix.csv` |
| `deepseek_v4_pro_limit1000` | `outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_deepseek_v4_pro_limit1000_eval_behavior_matrix.csv` |
| `deepseek_v4_pro_llama_limit1000` | `outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_deepseek_v4_pro_llama_limit1000_eval_behavior_matrix.csv` |

Main output folder:

```text
figures/psychometric_audit_benchmark/
```

This notebook does **not** overwrite the previous `figures/global_evaluator_audit/` outputs.


In [ ]:
from pathlib import Path
import json
import warnings
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import r2_score

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 240)

OUTDIR = Path('figures/psychometric_audit_benchmark')
OUTDIR.mkdir(parents=True, exist_ok=True)

FULL_CALE_VARIANT = 'full_attack_aware_cale'
MODEL_BASED_DIRECT_VARIANT = 'direct_llm_judge'
HEURISTIC_DIRECT_VARIANT = 'direct_trustllm_heuristic'

EXPECTED_TARGET_SPLITS = {'target_qwen_only', 'target_llama_only'}

VARIANT_ROLE = {
    'baseline_binary': 'baseline',
    'baseline_likert': 'baseline',
    'direct_trustllm_heuristic': 'direct',
    'direct_llm_judge': 'direct',
    'generic_cale': 'cale_family_generic',
    'attack_aware_cale': 'cale_family_attack_aware',
    'full_attack_aware_cale': 'cale_family_full',
}

# Core CALE dimensions used in the current thesis analysis.
CALE_DIMS = [
    'claim_status_recognition',
    'evidence_grounding',
    'source_faithfulness',
    'misinformation_detection',
    'hallucination_control',
    'uncertainty_handling',
    'framing_resistance',
    'correction_accuracy',
    'error_rejection',
]

FACTOR_GROUPS = {
    'factual_status': [
        'claim_status_recognition',
        'misinformation_detection',
        'error_rejection',
    ],
    'evidence_grounding_factor': [
        'evidence_grounding',
        'source_faithfulness',
        'hallucination_control',
    ],
    'epistemic_robustness': [
        'uncertainty_handling',
        'framing_resistance',
        'correction_accuracy',
    ],
}

RUNS = [
    {
        'run_id': 'heuristic_full',
        'evaluator_backend_family': 'heuristic',
        'evaluator_backend_model': 'heuristic/default',
        'evaluator_backend_label': 'Rule-based scoring backend (non-LLM)',
        'behavior_matrix_path': Path('outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_full_eval_behavior_matrix.csv'),
        'report_path': Path('outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_full_eval_report.json'),
        'expected_rows': 239976,
        'expected_variants': ['baseline_binary', 'baseline_likert', 'direct_trustllm_heuristic', 'generic_cale', 'attack_aware_cale', 'full_attack_aware_cale'],
        'expected_target_splits': ['target_qwen_only', 'target_llama_only'],
        'subset_policy': 'full_neutral_fever_qwen_and_llama_targets',
        'direct_baseline_variant': HEURISTIC_DIRECT_VARIANT,
        'notes': 'Main full-sample rule-based result; primary source for PCA/CFA.',
    },
    {
        'run_id': 'qwen25_7b_limit1000',
        'evaluator_backend_family': 'hf_local',
        'evaluator_backend_model': 'Qwen/Qwen2.5-7B-Instruct',
        'evaluator_backend_label': 'Qwen2.5-7B local HF judge',
        'behavior_matrix_path': Path('outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_strong_qwen25_7b_limit1000_eval_behavior_matrix.csv'),
        'report_path': Path('outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_strong_qwen25_7b_limit1000_eval_report.json'),
        'expected_rows': 2000,
        'expected_variants': ['direct_llm_judge', 'full_attack_aware_cale'],
        'expected_target_splits': ['target_qwen_only'],
        'subset_policy': 'limit1000_qwen_target_from_fixed_response_file',
        'direct_baseline_variant': MODEL_BASED_DIRECT_VARIANT,
        'notes': 'Matched Qwen-target strong local evaluator subset.',
    },
    {
        'run_id': 'qwen25_7b_llama_limit1000',
        'evaluator_backend_family': 'hf_local',
        'evaluator_backend_model': 'Qwen/Qwen2.5-7B-Instruct',
        'evaluator_backend_label': 'Qwen2.5-7B local HF judge',
        'behavior_matrix_path': Path('outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_strong_qwen25_7b_llama_limit1000_eval_behavior_matrix.csv'),
        'report_path': Path('outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_strong_qwen25_7b_llama_limit1000_eval_report.json'),
        'expected_rows': 2000,
        'expected_variants': ['direct_llm_judge', 'full_attack_aware_cale'],
        'expected_target_splits': ['target_llama_only'],
        'subset_policy': 'matched_limit1000_llama_target_from_fixed_response_file',
        'direct_baseline_variant': MODEL_BASED_DIRECT_VARIANT,
        'notes': 'Matched Llama-target strong local evaluator subset.',
    },
    {
        'run_id': 'deepseek_v4_pro_limit1000',
        'evaluator_backend_family': 'api',
        'evaluator_backend_model': 'deepseek-v4-pro',
        'evaluator_backend_label': 'DeepSeek V4-Pro API judge',
        'behavior_matrix_path': Path('outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_deepseek_v4_pro_limit1000_eval_behavior_matrix.csv'),
        'report_path': Path('outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_deepseek_v4_pro_limit1000_eval_report.json'),
        'expected_rows': 2000,
        'expected_variants': ['direct_llm_judge', 'full_attack_aware_cale'],
        'expected_target_splits': ['target_qwen_only'],
        'subset_policy': 'limit1000_qwen_target_from_fixed_response_file',
        'direct_baseline_variant': MODEL_BASED_DIRECT_VARIANT,
        'notes': 'Matched Qwen-target strong API evaluator subset.',
    },
    {
        'run_id': 'deepseek_v4_pro_llama_limit1000',
        'evaluator_backend_family': 'api',
        'evaluator_backend_model': 'deepseek-v4-pro',
        'evaluator_backend_label': 'DeepSeek V4-Pro API judge',
        'behavior_matrix_path': Path('outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_deepseek_v4_pro_llama_limit1000_eval_behavior_matrix.csv'),
        'report_path': Path('outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_deepseek_v4_pro_llama_limit1000_eval_report.json'),
        'expected_rows': 2000,
        'expected_variants': ['direct_llm_judge', 'full_attack_aware_cale'],
        'expected_target_splits': ['target_llama_only'],
        'subset_policy': 'matched_limit1000_llama_target_from_fixed_response_file',
        'direct_baseline_variant': MODEL_BASED_DIRECT_VARIANT,
        'notes': 'Matched Llama-target strong API evaluator subset.',
    },
]

run_registry = pd.DataFrame([{k: str(v) if isinstance(v, Path) else v for k, v in run.items()} for run in RUNS])
run_registry.to_csv(OUTDIR / 'run_registry.csv', index=False)
display(run_registry)


## 1. Load previous behavior matrices and audit coverage

This section mirrors the file registry and naming conventions from the previous notebook. The important separation is:

- `target_model`: the model that generated the answer being evaluated.
- `evaluator_backend_model`: the judge backend/model.
- `evaluator_variant`: the evaluation protocol, for example direct judge or full CALE.


In [ ]:
def target_split_name(model_name):
    value = str(model_name).lower()
    if 'qwen' in value:
        return 'target_qwen_only'
    if 'llama' in value:
        return 'target_llama_only'
    return 'target_other'


def joined_unique(series):
    return ', '.join(sorted(series.dropna().astype(str).unique()))


def load_run(run):
    path = run['behavior_matrix_path']
    if not path.exists():
        return None
    df = pd.read_csv(path)
    df = df.copy()
    df['target_model'] = df['model_name'].astype(str) if 'model_name' in df.columns else 'unknown_target_model'
    df['target_split'] = df['target_model'].map(target_split_name)
    df['evaluator_variant'] = df['variant'].astype(str) if 'variant' in df.columns else 'unknown_variant'
    df['variant_role'] = df['evaluator_variant'].map(VARIANT_ROLE).fillna('other_or_unknown')
    for key in ['run_id', 'evaluator_backend_family', 'evaluator_backend_model', 'evaluator_backend_label', 'subset_policy', 'direct_baseline_variant']:
        df[key] = run[key]
    return df


def available_cale_dims(df):
    return [c for c in CALE_DIMS if c in df.columns]

matrices = {}
audit_rows = []
for run in RUNS:
    df = load_run(run)
    if df is not None:
        matrices[run['run_id']] = df
        variants = sorted(df['evaluator_variant'].dropna().astype(str).unique())
        target_splits = sorted(df['target_split'].dropna().astype(str).unique())
    else:
        variants = []
        target_splits = []
    audit_rows.append({
        'run_id': run['run_id'],
        'path': str(run['behavior_matrix_path']),
        'exists': run['behavior_matrix_path'].exists(),
        'rows': len(df) if df is not None else 0,
        'expected_rows': run['expected_rows'],
        'row_count_ok': (len(df) == run['expected_rows']) if df is not None else False,
        'evaluator_backend_label': run['evaluator_backend_label'],
        'evaluator_backend_model': run['evaluator_backend_model'],
        'evaluator_variants': ', '.join(variants),
        'missing_expected_variants': ', '.join(sorted(set(run['expected_variants']) - set(variants))),
        'target_splits_present': ', '.join(target_splits),
        'missing_expected_target_splits': ', '.join(sorted(set(run['expected_target_splits']) - set(target_splits))),
        'available_cale_dims': ', '.join(available_cale_dims(df)) if df is not None else '',
        'notes': run['notes'],
    })

audit = pd.DataFrame(audit_rows)
audit.to_csv(OUTDIR / '00_input_file_coverage_audit.csv', index=False)
display(audit)

combined = pd.concat(matrices.values(), ignore_index=True, sort=False) if matrices else pd.DataFrame()
if not combined.empty:
    combined.to_csv(OUTDIR / '01_combined_available_behavior_matrix.csv', index=False)
    print('Combined rows:', len(combined))
    print('Columns:', len(combined.columns))
else:
    print('No behavior matrix files found. Run this notebook from the project root that contains outputs/small_models_all/.')


## 2. Prepare full CALE analysis table

For CFA and latent profiles, we focus on `full_attack_aware_cale`, because this is the richest structured evaluator output.


In [ ]:
if combined.empty:
    full_cale = pd.DataFrame()
else:
    full_cale = combined[combined['evaluator_variant'].eq(FULL_CALE_VARIANT)].copy()

missing_dims = [c for c in CALE_DIMS if c not in full_cale.columns]
print('Full CALE rows:', len(full_cale))
print('Missing CALE dimensions:', missing_dims)

for c in CALE_DIMS:
    if c in full_cale.columns:
        full_cale[c] = pd.to_numeric(full_cale[c], errors='coerce')

full_cale_quality = []
if not full_cale.empty:
    for keys, g in full_cale.groupby(['evaluator_backend_label', 'evaluator_backend_model', 'target_split'], dropna=False):
        row = dict(zip(['evaluator_backend_label', 'evaluator_backend_model', 'target_split'], keys))
        row['rows'] = len(g)
        row['target_models'] = joined_unique(g['target_model']) if 'target_model' in g.columns else ''
        row['complete_cale_dim_rows'] = g[CALE_DIMS].dropna().shape[0]
        row['complete_cale_dim_rate'] = row['complete_cale_dim_rows'] / max(row['rows'], 1)
        full_cale_quality.append(row)

full_cale_quality = pd.DataFrame(full_cale_quality)
full_cale_quality.to_csv(OUTDIR / '02_full_cale_data_quality_by_backend_split.csv', index=False)
display(full_cale_quality)


## 3. PCA internal structure diagnostic

PCA is used here as an exploratory internal-structure diagnostic. It does not prove validity. It tells us whether CALE dimensions are dominated by one general axis or whether there is residual multidimensional structure.


In [ ]:
def run_pca_table(df, dims, min_rows=10):
    rows = []
    loadings_rows = []
    score_frames = []
    group_cols = ['evaluator_backend_label', 'evaluator_backend_model']
    if df.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    for keys, g0 in df.groupby(group_cols, dropna=False):
        backend_label, backend_model = keys
        split_frames = [('pooled_available', g0)]
        if 'target_split' in g0.columns:
            split_frames.extend(list(g0.groupby('target_split', dropna=False)))
        for split_name, g in split_frames:
            use = g[dims].apply(pd.to_numeric, errors='coerce').dropna()
            if len(use) < min_rows:
                rows.append({
                    'evaluator_backend_label': backend_label,
                    'evaluator_backend_model': backend_model,
                    'target_split': split_name,
                    'rows': len(g),
                    'complete_rows': len(use),
                    'error': 'too few complete rows',
                })
                continue
            scaler = StandardScaler()
            X = scaler.fit_transform(use)
            n_comp = min(len(dims), X.shape[0], 9)
            pca = PCA(n_components=n_comp, random_state=0)
            scores = pca.fit_transform(X)
            explained = pca.explained_variance_ratio_
            rows.append({
                'evaluator_backend_label': backend_label,
                'evaluator_backend_model': backend_model,
                'target_split': split_name,
                'rows': len(g),
                'complete_rows': len(use),
                'n_dimensions': len(dims),
                'pc1_variance': explained[0],
                'pc1_pc2_cumulative_variance': explained[:2].sum(),
                'pc1_pc3_cumulative_variance': explained[:3].sum(),
                'pc1_pc4_cumulative_variance': explained[:4].sum() if len(explained) >= 4 else explained.sum(),
                'error': '',
            })
            for comp_i in range(n_comp):
                for dim, loading in zip(dims, pca.components_[comp_i]):
                    loadings_rows.append({
                        'evaluator_backend_label': backend_label,
                        'evaluator_backend_model': backend_model,
                        'target_split': split_name,
                        'component': f'PC{comp_i+1}',
                        'dimension': dim,
                        'loading': loading,
                        'abs_loading': abs(loading),
                    })
            score_df = g.loc[use.index, ['run_id', 'target_model', 'target_split', 'evaluator_backend_label', 'evaluator_backend_model']].copy()
            for i in range(min(4, scores.shape[1])):
                score_df[f'PC{i+1}'] = scores[:, i]
            score_frames.append(score_df)
    return pd.DataFrame(rows), pd.DataFrame(loadings_rows), pd.concat(score_frames, ignore_index=True, sort=False) if score_frames else pd.DataFrame()

pca_summary, pca_loadings, pca_scores = run_pca_table(full_cale, CALE_DIMS)
pca_summary.to_csv(OUTDIR / '03_pca_full_cale_summary.csv', index=False)
pca_loadings.to_csv(OUTDIR / '03_pca_full_cale_loadings.csv', index=False)
pca_scores.to_csv(OUTDIR / '03_pca_full_cale_scores.csv', index=False)
display(pca_summary.round(3))


In [ ]:
if not pca_summary.empty:
    plot_df = pca_summary[pca_summary['target_split'].eq('pooled_available')].copy()
    if not plot_df.empty:
        labels = plot_df['evaluator_backend_label'].astype(str)
        x = np.arange(len(plot_df))
        width = 0.35
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.bar(x - width/2, plot_df['pc1_variance'], width, label='PC1')
        ax.bar(x + width/2, plot_df['pc1_pc4_cumulative_variance'], width, label='PC1-PC4')
        ax.set_ylim(0, 1)
        ax.set_ylabel('Explained variance')
        ax.set_title('Full CALE PCA: General Factor vs Multidimensional Structure')
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=25, ha='right')
        ax.legend()
        fig.tight_layout()
        fig.savefig(OUTDIR / '03_pca_full_cale_pc1_pc4.png', dpi=200, bbox_inches='tight')
        plt.show()


## 4. CFA measurement models

We test three theory-driven CFA structures:

1. **One-factor model**: all CALE dimensions reflect one general evaluator quality factor.
2. **Three correlated factors**: factual status, evidence grounding, and epistemic robustness.
3. **Bifactor model**: one general factor plus three domain-specific factors.

Important: CFA tests whether the outputs conform to a proposed measurement model. It does not discover validity automatically.


In [ ]:
# semopy is the most convenient Python package for CFA/SEM.
# If it is not installed on the server, uncomment the next line once:
# %pip install semopy

try:
    import semopy
    from semopy import Model, calc_stats
    SEMOPY_AVAILABLE = True
    print('semopy version:', getattr(semopy, '__version__', 'unknown'))
except Exception as exc:
    SEMOPY_AVAILABLE = False
    print('semopy is not available. CFA cells will be skipped until semopy is installed.')
    print('Import error:', exc)


In [ ]:
ONE_FACTOR_MODEL = """
general_quality =~ claim_status_recognition + evidence_grounding + source_faithfulness + misinformation_detection + hallucination_control + uncertainty_handling + framing_resistance + correction_accuracy + error_rejection
"""

THREE_FACTOR_MODEL = """
factual_status =~ claim_status_recognition + misinformation_detection + error_rejection
evidence_grounding_factor =~ evidence_grounding + source_faithfulness + hallucination_control
epistemic_robustness =~ uncertainty_handling + framing_resistance + correction_accuracy

factual_status ~~ evidence_grounding_factor
factual_status ~~ epistemic_robustness
evidence_grounding_factor ~~ epistemic_robustness
"""

# Bifactor models can be numerically delicate with only three indicators per specific factor.
# We specify orthogonality between the general factor and group factors to keep the interpretation clean.
BIFACTOR_MODEL = """
general_quality =~ claim_status_recognition + evidence_grounding + source_faithfulness + misinformation_detection + hallucination_control + uncertainty_handling + framing_resistance + correction_accuracy + error_rejection
factual_status =~ claim_status_recognition + misinformation_detection + error_rejection
evidence_grounding_factor =~ evidence_grounding + source_faithfulness + hallucination_control
epistemic_robustness =~ uncertainty_handling + framing_resistance + correction_accuracy

general_quality ~~ 0*factual_status
general_quality ~~ 0*evidence_grounding_factor
general_quality ~~ 0*epistemic_robustness
factual_status ~~ 0*evidence_grounding_factor
factual_status ~~ 0*epistemic_robustness
evidence_grounding_factor ~~ 0*epistemic_robustness
"""

CFA_MODELS = {
    'one_factor': ONE_FACTOR_MODEL,
    'three_correlated_factors': THREE_FACTOR_MODEL,
    'bifactor': BIFACTOR_MODEL,
}

print(THREE_FACTOR_MODEL)


In [ ]:
def standardize_cfa_input(df, dims, max_rows=20000, random_state=42):
    use = df[dims].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).dropna().copy()
    # Remove zero-variance columns locally.
    variances = use.var(axis=0)
    keep = [c for c in dims if c in use.columns and variances.get(c, 0) > 1e-12]
    use = use[keep]
    if max_rows is not None and len(use) > max_rows:
        use = use.sample(max_rows, random_state=random_state)
    # semopy can fit raw data. Standardizing improves comparability of loadings.
    use = (use - use.mean()) / use.std(ddof=0)
    use = use.replace([np.inf, -np.inf], np.nan).dropna()
    return use


def flatten_semopy_stats(stats):
    if stats is None:
        return {}
    if isinstance(stats, pd.DataFrame):
        if 'Value' in stats.columns:
            return stats['Value'].to_dict()
        if len(stats) == 1:
            return stats.iloc[0].to_dict()
    if isinstance(stats, dict):
        return stats
    return {}


def fit_cfa_model(data, model_desc):
    if not SEMOPY_AVAILABLE:
        return None, {}, pd.DataFrame(), 'semopy_not_available'
    try:
        model = Model(model_desc)
        model.fit(data)
        stats = flatten_semopy_stats(calc_stats(model))
        estimates = model.inspect(std_est=True)
        return model, stats, estimates, ''
    except Exception as exc:
        return None, {}, pd.DataFrame(), repr(exc)


def get_stat(stats, *names):
    lowered = {str(k).lower(): v for k, v in stats.items()}
    for name in names:
        if name in stats:
            return stats[name]
        if name.lower() in lowered:
            return lowered[name.lower()]
    return np.nan


def cfa_fit_rows_for_group(group_name, group_df, group_meta, dims=CALE_DIMS):
    rows = []
    loading_rows = []
    factor_score_frames = []
    data = standardize_cfa_input(group_df, dims)
    if len(data) < 50:
        base = {**group_meta, 'analysis_group': group_name, 'complete_rows': len(data)}
        for model_name in CFA_MODELS:
            rows.append({**base, 'cfa_model': model_name, 'error': 'too few complete rows for CFA'})
        return rows, loading_rows, factor_score_frames
    for model_name, desc in CFA_MODELS.items():
        model, stats, estimates, error = fit_cfa_model(data, desc)
        row = {
            **group_meta,
            'analysis_group': group_name,
            'cfa_model': model_name,
            'complete_rows': len(data),
            'chi2': get_stat(stats, 'chi2', 'Chi-square'),
            'df': get_stat(stats, 'DoF', 'df'),
            'p_value': get_stat(stats, 'chi2 p-value', 'p-value'),
            'cfi': get_stat(stats, 'CFI'),
            'tli': get_stat(stats, 'TLI'),
            'rmsea': get_stat(stats, 'RMSEA'),
            'srmr': get_stat(stats, 'SRMR'),
            'aic': get_stat(stats, 'AIC'),
            'bic': get_stat(stats, 'BIC'),
            'error': error,
        }
        rows.append(row)
        if estimates is not None and not estimates.empty:
            est = estimates.copy()
            est['analysis_group'] = group_name
            est['cfa_model'] = model_name
            for k, v in group_meta.items():
                est[k] = v
            loading_rows.append(est)
        if model is not None:
            try:
                factor_scores = model.predict_factors(data).reset_index(drop=True)
                meta = group_df.loc[data.index, [c for c in ['run_id', 'id', 'target_model', 'target_split', 'reference_label', 'evaluator_backend_label', 'evaluator_backend_model'] if c in group_df.columns]].reset_index(drop=True)
                fs = pd.concat([meta, factor_scores], axis=1)
                fs['analysis_group'] = group_name
                fs['cfa_model'] = model_name
                for k, v in group_meta.items():
                    if k not in fs.columns:
                        fs[k] = v
                factor_score_frames.append(fs)
            except Exception as exc:
                pass
    return rows, loading_rows, factor_score_frames


In [ ]:
cfa_rows = []
cfa_loading_frames = []
cfa_factor_score_frames = []

if full_cale.empty:
    print('No full CALE data available.')
elif not SEMOPY_AVAILABLE:
    print('Install semopy to run CFA: %pip install semopy')
else:
    for keys, g in full_cale.groupby(['evaluator_backend_label', 'evaluator_backend_model'], dropna=False):
        backend_label, backend_model = keys
        meta = {
            'evaluator_backend_label': backend_label,
            'evaluator_backend_model': backend_model,
            'target_split': 'pooled_available',
            'rows_raw': len(g),
            'target_models': joined_unique(g['target_model']) if 'target_model' in g.columns else '',
        }
        rows, loads, scores = cfa_fit_rows_for_group('backend_pooled', g, meta)
        cfa_rows.extend(rows)
        cfa_loading_frames.extend(loads)
        cfa_factor_score_frames.extend(scores)

        for split, gs in g.groupby('target_split', dropna=False):
            meta_split = {
                'evaluator_backend_label': backend_label,
                'evaluator_backend_model': backend_model,
                'target_split': split,
                'rows_raw': len(gs),
                'target_models': joined_unique(gs['target_model']) if 'target_model' in gs.columns else '',
            }
            rows, loads, scores = cfa_fit_rows_for_group('backend_by_target_split', gs, meta_split)
            cfa_rows.extend(rows)
            cfa_loading_frames.extend(loads)
            cfa_factor_score_frames.extend(scores)

cfa_fit = pd.DataFrame(cfa_rows)
cfa_loadings = pd.concat(cfa_loading_frames, ignore_index=True, sort=False) if cfa_loading_frames else pd.DataFrame()
cfa_factor_scores = pd.concat(cfa_factor_score_frames, ignore_index=True, sort=False) if cfa_factor_score_frames else pd.DataFrame()

cfa_fit.to_csv(OUTDIR / '04_cfa_model_fit_by_backend_split.csv', index=False)
cfa_loadings.to_csv(OUTDIR / '04_cfa_parameter_estimates.csv', index=False)
cfa_factor_scores.to_csv(OUTDIR / '04_cfa_factor_scores_long.csv', index=False)

display(cfa_fit.sort_values(['evaluator_backend_label', 'target_split', 'cfa_model']).round(4))


In [ ]:
# Plot CFA fit for pooled backend models.
if not cfa_fit.empty:
    pooled = cfa_fit[cfa_fit['target_split'].eq('pooled_available') & cfa_fit['error'].fillna('').eq('')].copy()
    if not pooled.empty and 'cfi' in pooled.columns:
        pivot = pooled.pivot_table(index='evaluator_backend_label', columns='cfa_model', values='cfi', aggfunc='first')
        pivot.to_csv(OUTDIR / '04_cfa_cfi_pivot_pooled.csv')
        display(pivot.round(3))
        ax = pivot.plot(kind='bar', figsize=(10,5))
        ax.set_ylim(0, 1)
        ax.set_ylabel('CFI')
        ax.set_title('CFA model fit by evaluator backend: CFI')
        ax.tick_params(axis='x', rotation=25)
        fig = ax.get_figure()
        fig.tight_layout()
        fig.savefig(OUTDIR / '04_cfa_cfi_by_backend.png', dpi=200, bbox_inches='tight')
        plt.show()


## 5. Reliability diagnostics

These are descriptive reliability checks for the CALE dimensions and proposed subdomains. They complement CFA but do not replace validation.


In [ ]:
def cronbach_alpha(df):
    X = df.apply(pd.to_numeric, errors='coerce').dropna()
    k = X.shape[1]
    if k < 2 or len(X) < 3:
        return np.nan
    item_vars = X.var(axis=0, ddof=1).sum()
    total_var = X.sum(axis=1).var(ddof=1)
    if total_var <= 1e-12:
        return np.nan
    return (k / (k - 1)) * (1 - item_vars / total_var)

reliability_rows = []
if not full_cale.empty:
    for keys, g in full_cale.groupby(['evaluator_backend_label', 'evaluator_backend_model'], dropna=False):
        backend_label, backend_model = keys
        for split_name, gs in [('pooled_available', g)] + list(g.groupby('target_split', dropna=False)):
            reliability_rows.append({
                'evaluator_backend_label': backend_label,
                'evaluator_backend_model': backend_model,
                'target_split': split_name,
                'scale': 'all_cale_dimensions',
                'n_items': len(CALE_DIMS),
                'rows_raw': len(gs),
                'complete_rows': gs[CALE_DIMS].dropna().shape[0],
                'cronbach_alpha': cronbach_alpha(gs[CALE_DIMS]),
            })
            for factor_name, dims in FACTOR_GROUPS.items():
                reliability_rows.append({
                    'evaluator_backend_label': backend_label,
                    'evaluator_backend_model': backend_model,
                    'target_split': split_name,
                    'scale': factor_name,
                    'n_items': len(dims),
                    'rows_raw': len(gs),
                    'complete_rows': gs[dims].dropna().shape[0],
                    'cronbach_alpha': cronbach_alpha(gs[dims]),
                })

reliability = pd.DataFrame(reliability_rows)
reliability.to_csv(OUTDIR / '05_reliability_cronbach_alpha.csv', index=False)
display(reliability.round(3))


## 6. Construct-relevant and nuisance signal construction

This section builds a dynamic signal registry from columns that actually exist in the behavior matrices.

- **Construct-relevant signals**: proxy variables that should theoretically align with evaluator quality or specific CALE factors.
- **Nuisance signals**: variables that should not strongly drive evaluator scores, such as response length, target model identity, or label frequency.

If you later add human labels, put them into the `CONSTRUCT_RELEVANT_CANDIDATES` list.


In [ ]:
def first_existing(columns, candidates):
    for c in candidates:
        if c in columns:
            return c
    return None


def add_text_length_features(df):
    df = df.copy()
    text_candidates = [
        'response', 'model_response', 'generated_response', 'answer', 'prediction',
        'candidate_response', 'output', 'completion', 'text'
    ]
    text_col = first_existing(df.columns, text_candidates)
    if text_col is not None:
        df['nuisance_response_length_chars'] = df[text_col].astype(str).str.len()
        df['nuisance_response_length_words'] = df[text_col].astype(str).str.split().map(len)
        print('Using text column for length nuisance features:', text_col)
    else:
        print('No response text column found for length nuisance features.')
    return df


def add_label_and_target_nuisance_features(df):
    df = df.copy()
    if 'reference_label' in df.columns:
        dummies = pd.get_dummies(df['reference_label'].astype(str), prefix='nuisance_reference_label')
        df = pd.concat([df, dummies], axis=1)
    if 'target_split' in df.columns:
        dummies = pd.get_dummies(df['target_split'].astype(str), prefix='nuisance_target_split')
        df = pd.concat([df, dummies], axis=1)
    if 'target_model' in df.columns:
        dummies = pd.get_dummies(df['target_model'].astype(str), prefix='nuisance_target_model')
        df = pd.concat([df, dummies], axis=1)
    return df

analysis_df = add_text_length_features(full_cale)
analysis_df = add_label_and_target_nuisance_features(analysis_df)

CONSTRUCT_RELEVANT_CANDIDATES = [
    # Existing proxy columns from the earlier behavior-matrix code, if present.
    'proxy_nei_uncertainty_failure',
    'proxy_refutes_correction_credit',
    'proxy_supports_status_failure',
    'nei_uncertainty_failure',
    'refutes_correction_credit',
    'supports_status_failure',
    # Possible future human/external signals.
    'human_factuality_score',
    'human_preference_score',
    'human_correctness',
    'gold_correctness',
    'label_correctness',
    'attack_success',
    'attack_resistant',
    'evidence_support_score',
    'contradiction_score',
    'unsupported_claim_count',
]

construct_relevant_cols = [c for c in CONSTRUCT_RELEVANT_CANDIDATES if c in analysis_df.columns]
nuisance_cols = [c for c in analysis_df.columns if c.startswith('nuisance_')]

signal_registry = pd.DataFrame([
    {'signal_type': 'construct_relevant_proxy', 'column': c} for c in construct_relevant_cols
] + [
    {'signal_type': 'nuisance', 'column': c} for c in nuisance_cols
])

signal_registry.to_csv(OUTDIR / '06_signal_registry.csv', index=False)
display(signal_registry)


## 7. Convergent and discriminant diagnostics

Here we correlate CFA factor scores with construct-relevant proxies and nuisance variables.

Interpretation:

- Higher association with construct-relevant signals supports **convergent evidence**.
- Lower association with nuisance signals supports **discriminant evidence**.

This section is proxy-based unless human validation columns are added.


In [ ]:
def safe_corr(a, b):
    x = pd.to_numeric(a, errors='coerce')
    y = pd.to_numeric(b, errors='coerce')
    m = x.notna() & y.notna()
    if m.sum() < 5 or x[m].nunique() < 2 or y[m].nunique() < 2:
        return np.nan
    return x[m].corr(y[m])

# Use factor scores from the best-theory model. If unavailable, fall back to PCA scores, then final_score.
PREFERRED_CFA_MODEL = 'three_correlated_factors'

if not cfa_factor_scores.empty and 'cfa_model' in cfa_factor_scores.columns:
    latent_df = cfa_factor_scores[cfa_factor_scores['cfa_model'].eq(PREFERRED_CFA_MODEL)].copy()
    factor_cols = [c for c in ['general_quality', 'factual_status', 'evidence_grounding_factor', 'epistemic_robustness'] if c in latent_df.columns]
    score_source = f'CFA factor scores: {PREFERRED_CFA_MODEL}'
else:
    latent_df = pd.DataFrame()
    factor_cols = []
    score_source = 'none'

if latent_df.empty and not pca_scores.empty:
    latent_df = pca_scores.copy()
    factor_cols = [c for c in ['PC1', 'PC2', 'PC3', 'PC4'] if c in latent_df.columns]
    score_source = 'PCA scores fallback'

if latent_df.empty and not analysis_df.empty and 'final_score' in analysis_df.columns:
    latent_df = analysis_df.copy()
    factor_cols = ['final_score']
    score_source = 'final_score fallback'

print('Score source:', score_source)
print('Score columns:', factor_cols)

# Attach available signal columns by a robust key.
merge_keys = [c for c in ['run_id', 'id', 'target_model', 'target_split', 'reference_label', 'evaluator_backend_label', 'evaluator_backend_model'] if c in latent_df.columns and c in analysis_df.columns]
if not latent_df.empty and merge_keys:
    signals_to_merge = merge_keys + construct_relevant_cols + nuisance_cols
    signals_to_merge = list(dict.fromkeys([c for c in signals_to_merge if c in analysis_df.columns]))
    latent_signals = latent_df.merge(analysis_df[signals_to_merge], on=merge_keys, how='left')
else:
    latent_signals = latent_df.copy()

corr_rows = []
for backend, g in latent_signals.groupby('evaluator_backend_label', dropna=False) if 'evaluator_backend_label' in latent_signals.columns else []:
    for score_col in factor_cols:
        for signal_col in construct_relevant_cols:
            if signal_col in g.columns:
                corr_rows.append({
                    'evaluator_backend_label': backend,
                    'score_source': score_source,
                    'score_column': score_col,
                    'signal_type': 'construct_relevant_proxy',
                    'signal_column': signal_col,
                    'n': g[[score_col, signal_col]].dropna().shape[0],
                    'correlation': safe_corr(g[score_col], g[signal_col]),
                })
        for signal_col in nuisance_cols:
            if signal_col in g.columns:
                corr_rows.append({
                    'evaluator_backend_label': backend,
                    'score_source': score_source,
                    'score_column': score_col,
                    'signal_type': 'nuisance',
                    'signal_column': signal_col,
                    'n': g[[score_col, signal_col]].dropna().shape[0],
                    'correlation': safe_corr(g[score_col], g[signal_col]),
                })

validity_correlations = pd.DataFrame(corr_rows)
if not validity_correlations.empty:
    validity_correlations['abs_correlation'] = validity_correlations['correlation'].abs()
validity_correlations.to_csv(OUTDIR / '07_convergent_discriminant_correlations.csv', index=False)
display(validity_correlations.sort_values(['evaluator_backend_label', 'score_column', 'signal_type', 'abs_correlation'], ascending=[True, True, True, False]).head(80))


In [ ]:
# Regression-style diagnostic: score ~ construct-relevant proxies + nuisance proxies.
# This is descriptive; with many dummy variables and small samples, interpret cautiously.
regression_rows = []
if not latent_signals.empty:
    for backend, g in latent_signals.groupby('evaluator_backend_label', dropna=False):
        for score_col in factor_cols:
            y = pd.to_numeric(g[score_col], errors='coerce')
            predictors_construct = [c for c in construct_relevant_cols if c in g.columns]
            predictors_nuisance = [c for c in nuisance_cols if c in g.columns]
            for block_name, predictors in [
                ('construct_only', predictors_construct),
                ('nuisance_only', predictors_nuisance),
                ('construct_plus_nuisance', predictors_construct + predictors_nuisance),
            ]:
                if not predictors:
                    continue
                X = g[predictors].apply(pd.to_numeric, errors='coerce')
                data = pd.concat([y.rename(score_col), X], axis=1).replace([np.inf, -np.inf], np.nan).dropna()
                # Avoid too many predictors for too few rows.
                if len(data) < max(20, 3 * (len(predictors) + 1)):
                    regression_rows.append({
                        'evaluator_backend_label': backend,
                        'score_column': score_col,
                        'predictor_block': block_name,
                        'n': len(data),
                        'n_predictors': len(predictors),
                        'r2': np.nan,
                        'error': 'too few rows for predictor count',
                    })
                    continue
                model = LinearRegression()
                model.fit(data[predictors], data[score_col])
                pred = model.predict(data[predictors])
                regression_rows.append({
                    'evaluator_backend_label': backend,
                    'score_column': score_col,
                    'predictor_block': block_name,
                    'n': len(data),
                    'n_predictors': len(predictors),
                    'r2': r2_score(data[score_col], pred),
                    'error': '',
                })

validity_regression = pd.DataFrame(regression_rows)
validity_regression.to_csv(OUTDIR / '07_validity_signal_regression_blocks.csv', index=False)
display(validity_regression.round(3))


## 8. Psychometric audit score

This is a compact dashboard score for comparing evaluator backends. It is intentionally transparent and adjustable.

Components:

1. **Internal structure score** from CFA fit.
2. **Convergent score** from construct-relevant proxy correlations.
3. **Discriminant score** from low nuisance correlations.
4. **Invariance score** from target-split stability of CFA fit.

This is an audit score, not a proof of validity.


In [ ]:
def scale_01(x, low, high, reverse=False):
    if pd.isna(x):
        return np.nan
    z = (x - low) / (high - low)
    z = max(0, min(1, z))
    return 1 - z if reverse else z

# Internal structure score based on three-factor model by default.
internal_rows = []
if not cfa_fit.empty:
    use_fit = cfa_fit[(cfa_fit['target_split'].eq('pooled_available')) & (cfa_fit['cfa_model'].eq(PREFERRED_CFA_MODEL))].copy()
    for _, r in use_fit.iterrows():
        cfi_score = scale_01(r.get('cfi', np.nan), 0.80, 0.98)
        tli_score = scale_01(r.get('tli', np.nan), 0.80, 0.98)
        rmsea_score = scale_01(r.get('rmsea', np.nan), 0.02, 0.10, reverse=True)
        srmr_score = scale_01(r.get('srmr', np.nan), 0.02, 0.10, reverse=True)
        vals = [v for v in [cfi_score, tli_score, rmsea_score, srmr_score] if not pd.isna(v)]
        internal_rows.append({
            'evaluator_backend_label': r['evaluator_backend_label'],
            'internal_structure_score': np.mean(vals) if vals else np.nan,
            'cfi': r.get('cfi', np.nan),
            'tli': r.get('tli', np.nan),
            'rmsea': r.get('rmsea', np.nan),
            'srmr': r.get('srmr', np.nan),
        })
internal_score = pd.DataFrame(internal_rows)

# Convergent/discriminant scores from correlations.
if not validity_correlations.empty:
    convergent_score = (
        validity_correlations[validity_correlations['signal_type'].eq('construct_relevant_proxy')]
        .groupby('evaluator_backend_label')['abs_correlation'].mean()
        .reset_index(name='convergent_score_raw')
    )
    nuisance_penalty = (
        validity_correlations[validity_correlations['signal_type'].eq('nuisance')]
        .groupby('evaluator_backend_label')['abs_correlation'].mean()
        .reset_index(name='nuisance_association_raw')
    )
else:
    convergent_score = pd.DataFrame(columns=['evaluator_backend_label', 'convergent_score_raw'])
    nuisance_penalty = pd.DataFrame(columns=['evaluator_backend_label', 'nuisance_association_raw'])

# Invariance proxy: smaller fit-index instability across target splits is better.
invariance_rows = []
if not cfa_fit.empty:
    split_fit = cfa_fit[(cfa_fit['analysis_group'].eq('backend_by_target_split')) & (cfa_fit['cfa_model'].eq(PREFERRED_CFA_MODEL))].copy()
    for backend, g in split_fit.groupby('evaluator_backend_label', dropna=False):
        if g['target_split'].nunique() < 2:
            invariance_rows.append({'evaluator_backend_label': backend, 'invariance_score': np.nan, 'fit_instability': np.nan, 'note': 'less than two target splits'})
            continue
        cols = [c for c in ['cfi', 'rmsea', 'srmr'] if c in g.columns]
        instability_values = []
        for c in cols:
            vals = pd.to_numeric(g[c], errors='coerce').dropna()
            if len(vals) >= 2:
                instability_values.append(vals.max() - vals.min())
        fit_instability = np.mean(instability_values) if instability_values else np.nan
        invariance_rows.append({
            'evaluator_backend_label': backend,
            'invariance_score': scale_01(fit_instability, 0.00, 0.20, reverse=True),
            'fit_instability': fit_instability,
            'note': 'proxy based on target-split fit-index stability',
        })
invariance_score = pd.DataFrame(invariance_rows)

benchmark = internal_score.merge(convergent_score, on='evaluator_backend_label', how='outer')
benchmark = benchmark.merge(nuisance_penalty, on='evaluator_backend_label', how='outer')
benchmark = benchmark.merge(invariance_score, on='evaluator_backend_label', how='outer')

if not benchmark.empty:
    # Convert raw convergent correlation into bounded score. Stronger is better.
    benchmark['convergent_score'] = benchmark['convergent_score_raw'].apply(lambda x: scale_01(x, 0.00, 0.50))
    # Discriminant is good when nuisance association is low.
    benchmark['discriminant_score'] = benchmark['nuisance_association_raw'].apply(lambda x: scale_01(x, 0.00, 0.40, reverse=True))
    score_cols = ['internal_structure_score', 'convergent_score', 'discriminant_score', 'invariance_score']
    benchmark['psychometric_audit_score'] = benchmark[score_cols].mean(axis=1, skipna=True)

benchmark.to_csv(OUTDIR / '08_psychometric_audit_benchmark_score.csv', index=False)
display(benchmark.sort_values('psychometric_audit_score', ascending=False).round(3) if not benchmark.empty else benchmark)


## 9. Target-model evaluator-mediated ability profiles

This section uses latent factor scores to summarize the evaluated models. These are **evaluator-mediated ability profiles**, not ground-truth abilities.

Best interpretation:

> Given this evaluator and this measurement model, how does each target model profile across factual status, evidence grounding, and epistemic robustness?


In [ ]:
profile_source = latent_df.copy() if not latent_df.empty else pd.DataFrame()
profile_factor_cols = [c for c in ['general_quality', 'factual_status', 'evidence_grounding_factor', 'epistemic_robustness', 'PC1', 'PC2', 'PC3', 'PC4', 'final_score'] if c in profile_source.columns]

ability_rows = []
if not profile_source.empty and profile_factor_cols:
    group_cols = [c for c in ['evaluator_backend_label', 'evaluator_backend_model', 'cfa_model', 'target_model', 'target_split'] if c in profile_source.columns]
    for keys, g in profile_source.groupby(group_cols, dropna=False):
        base = dict(zip(group_cols, keys)) if isinstance(keys, tuple) else {group_cols[0]: keys}
        for factor in profile_factor_cols:
            vals = pd.to_numeric(g[factor], errors='coerce').dropna()
            ability_rows.append({
                **base,
                'score_source': score_source,
                'ability_dimension': factor,
                'n': len(vals),
                'mean': vals.mean() if len(vals) else np.nan,
                'std': vals.std(ddof=1) if len(vals) > 1 else np.nan,
                'se': vals.std(ddof=1) / np.sqrt(len(vals)) if len(vals) > 1 else np.nan,
            })

ability_profiles = pd.DataFrame(ability_rows)
ability_profiles.to_csv(OUTDIR / '09_target_model_ability_profiles.csv', index=False)
display(ability_profiles.round(3))


In [ ]:
if not ability_profiles.empty:
    for backend, g in ability_profiles.groupby('evaluator_backend_label', dropna=False):
        for dim in g['ability_dimension'].dropna().unique():
            p = g[g['ability_dimension'].eq(dim)].copy()
            if p.empty or p['target_model'].nunique() < 1:
                continue
            p = p.sort_values('target_model')
            fig, ax = plt.subplots(figsize=(8, 4.5))
            x = np.arange(len(p))
            ax.bar(x, p['mean'])
            if 'se' in p.columns:
                ax.errorbar(x, p['mean'], yerr=1.96 * p['se'].fillna(0), fmt='none', capsize=4)
            ax.set_xticks(x)
            ax.set_xticklabels(p['target_model'].astype(str), rotation=25, ha='right')
            ax.set_ylabel('Latent score mean')
            ax.set_title(f'{backend}: target-model profile on {dim}')
            fig.tight_layout()
            safe_backend = ''.join(ch if ch.isalnum() else '_' for ch in str(backend))[:60]
            safe_dim = ''.join(ch if ch.isalnum() else '_' for ch in str(dim))
            fig.savefig(OUTDIR / f'09_ability_profile_{safe_backend}_{safe_dim}.png', dpi=200, bbox_inches='tight')
            plt.show()


## 10. Auto-generated interpretation tables

This section produces compact tables that can be copied into the thesis Results section. The text is deliberately conservative.


In [ ]:
interpretation_rows = []

if not pca_summary.empty:
    for _, r in pca_summary[pca_summary['target_split'].eq('pooled_available')].iterrows():
        interpretation_rows.append({
            'section': 'PCA internal structure',
            'evaluator_backend_label': r['evaluator_backend_label'],
            'finding': f"PC1 explains {r['pc1_variance']:.3f} of variance; PC1-PC4 explain {r['pc1_pc4_cumulative_variance']:.3f}.",
            'safe_interpretation': 'This indicates a shared evaluation signal if PC1 is substantial, while residual PC variance indicates non-redundant multidimensional structure.',
            'claim_boundary': 'PCA is internal-structure evidence, not external validity evidence.',
        })

if not cfa_fit.empty:
    best = cfa_fit[(cfa_fit['target_split'].eq('pooled_available')) & (cfa_fit['error'].fillna('').eq(''))].copy()
    if not best.empty:
        for backend, g in best.groupby('evaluator_backend_label'):
            # Prefer lowest BIC if available.
            gg = g.copy()
            gg['bic_numeric'] = pd.to_numeric(gg['bic'], errors='coerce')
            chosen = gg.sort_values('bic_numeric').iloc[0] if gg['bic_numeric'].notna().any() else gg.iloc[0]
            interpretation_rows.append({
                'section': 'CFA model comparison',
                'evaluator_backend_label': backend,
                'finding': f"Best available CFA model by BIC: {chosen['cfa_model']}.",
                'safe_interpretation': 'The evaluator backend is compared by how well its outputs conform to the proposed measurement structures.',
                'claim_boundary': 'A better CFA fit does not prove that the evaluator is more valid without external criteria.',
            })

if not benchmark.empty:
    for _, r in benchmark.sort_values('psychometric_audit_score', ascending=False).iterrows():
        interpretation_rows.append({
            'section': 'Psychometric audit benchmark',
            'evaluator_backend_label': r['evaluator_backend_label'],
            'finding': f"Audit score = {r.get('psychometric_audit_score', np.nan):.3f}.",
            'safe_interpretation': 'This combines internal structure, convergent proxy evidence, discriminant proxy evidence, and target-split stability.',
            'claim_boundary': 'The score is a transparent audit index, not a final validity ranking.',
        })

interpretation_table = pd.DataFrame(interpretation_rows)
interpretation_table.to_csv(OUTDIR / '10_safe_interpretation_table.csv', index=False)
display(interpretation_table)


## 11. Recommended thesis wording

Use this wording if the analysis runs successfully:

> We extend the CALE analysis from score comparison to a psychometric audit of evaluator behavior. PCA is first used as an exploratory internal-structure diagnostic. We then fit theory-driven CFA models to test whether CALE dimensions conform to a coherent measurement structure, comparing a one-factor model, a three-factor correlated model, and a bifactor model. Evaluator backends are compared not by raw score levels, but by the degree to which their structured judgments exhibit interpretable internal structure, convergence with construct-relevant proxy signals, separation from nuisance signals, and stability across target-model splits. Finally, latent factor scores are used to construct evaluator-mediated ability profiles of target models. These profiles should be interpreted as measurement-model-based summaries under a given evaluator, not as ground-truth abilities without external human validation.


## 12. Output checklist

After running the notebook, inspect these files:

| Output file | Purpose |
|---|---|
| `00_input_file_coverage_audit.csv` | confirms which previous result files exist and whether row counts match |
| `03_pca_full_cale_summary.csv` | PCA internal structure results |
| `04_cfa_model_fit_by_backend_split.csv` | CFA model fit comparison |
| `04_cfa_parameter_estimates.csv` | CFA loadings/residuals/parameter estimates |
| `04_cfa_factor_scores_long.csv` | latent factor scores for later profiling |
| `05_reliability_cronbach_alpha.csv` | scale/domain reliability checks |
| `06_signal_registry.csv` | available construct-relevant and nuisance signals |
| `07_convergent_discriminant_correlations.csv` | proxy-based validity diagnostics |
| `08_psychometric_audit_benchmark_score.csv` | compact evaluator audit score |
| `09_target_model_ability_profiles.csv` | evaluator-mediated target-model ability profiles |
| `10_safe_interpretation_table.csv` | conservative thesis interpretation snippets |
